**<h1 style="text-align: center;font-size: 3rem">Basic Exploratory Data Analysis</h1><h2 style="text-align: center;font-size: 1.3rem">(Notebook I)</h2>**


## Imports

Using Pandas and NumPy for numerical computation and exploration while using Seaborn and MatPlotLib to visualize data.


In [ ]:
import itertools
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv

from fraud_detection_pipeline.utils.constants import PARENT_DIR, RAW_DIRECTORY
from fraud_detection_pipeline.utils.displays import proportions_values_counts

## Setup

Loading the random state to seed into analysis for replicable results. The random state is obtained from the `.env` file from the variable named _'RANDOM_STATE'_.


In [ ]:
load_dotenv()
RANDOM_STATE = int(os.getenv("RANDOM_STATE", 0))

print(f"{RANDOM_STATE=}")

Setting the background on graphs to be darker for preference.


In [ ]:
plt.style.use("dark_background")

## Reading CSV into a DataFrame

As completed in the `01_data_ingestion.ipynb` notebook, the data is stored as a Parquet file, requiring the `read_parquet` method to read the data into a DataFrame object.


In [ ]:
transactions: pd.DataFrame = pd.read_csv(PARENT_DIR / RAW_DIRECTORY / "creditcard.csv")

## Characteristics of the Data

Observing the first and last 10 entries, observing the data's first-glance characteristics.


In [ ]:
transactions.head(10)

In [ ]:
transactions.tail(10)

As mentioned per the Content section of the data's [_Kaggle_ page](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud), the time appears to represent the number of seconds a transaction occured after the first record transaction and 28 of the variables are abstracted including their original name and nature.

Due to PCA transformation and other privacy policies, the original names and natures of these variables are unknown. These still describe the transaction's characteristics and can still be potentially explored and implemented into future models.

The currency amount is measured is not clearly stated or evident in the Kaggle page or the data, however, due to the data being sourced in Europe it's possible it is measured in Euros. This is not hugely relevant to the analysis however as the scale in difference between amounts is what matters.

The class is either _0_ for a genuine transaction or _1_ for a fraudulent transaction.


## More Insight

Using the DataFrame's `describe()` and `info()` methods, deeper characteristics of the columns begin to show.


In [ ]:
transactions.describe()

At a first glance, not much can be interpreted from the described columns. The distributions of the unnamed variables are all very similar as they are centered around a near 0 value. This could hint that the redacted variables were normalized prior to being placed into the dataset, using _'Min-Max Normalization'_ or another type of normalization.


In [ ]:
transactions.info()

There is no missing data in the non-modified dataset as the Non-Null Counts of each column is equal to the number of rows. Aside from the _'time'_ and _'is_fraud'_ columns which are measured as integers and booleans respectively, all other columns are continuous values stored as floats.


## Class Proportions

Better understanding the proportion of the classes (genuine or fraudulent).


In [ ]:
target_name: str = "Class"
print(transactions.value_counts(target_name))
print(proportions_values_counts(transactions[target_name]))

There is severe class imbalance, being that less than 0.17% of transaction in the data are fraudulent. Resampling techniques like SMOTE will likely be utilized and evaluated against other techniques along with training to focus more on making less false-fraudulent labeling.


## Visualizations

The target and features are separated to simplify visualization and to group and plot rows by their class.


In [ ]:
target = transactions[target_name]
features = transactions.drop(columns=[target_name])

The values are all standardized to standardize axes.


In [ ]:
standardized_features = (features - features.mean()) / features.std()

Storing the number of rows and columns for use in generating visualization.


In [ ]:
row_count = transactions.shape[0]
col_count = transactions.shape[1]

As there are too many entries to plot, 40% of the entries are used for visualization, stratified by the target.


In [ ]:
grouped_transactions = transactions.groupby(target_name, group_keys=False)
sampled_transactions = grouped_transactions.sample(frac=0.4, random_state=RANDOM_STATE)

A sanity check to make sure the proportion of the sampled fraudulent and genuine transactions still match the entire dataset.


In [ ]:
print(proportions_values_counts(sampled_transactions[target_name]))

### Correlation Matrices

Getting two DataFrames from the sampled transactions with one being only the fraudulent transactions and the other being only the genuine transaction. Each of the correlations are calculated using the `.corr()` DataFrame method.


In [ ]:
class_0_df = transactions[transactions[target_name] == 0]
class_0_corr = class_0_df.corr()

class_1_df = transactions[transactions[target_name] == 1]
class_1_corr = class_1_df.corr()

Creating a subplot with two axes for plots. A heatmap is placed in each available axis to visualize the correlation matrices. The `diag_mask` indicates to the heatmap to ignore the correlations that form a diagonal of 1.00s (a result of a variable being 100% correlated with itself).


In [ ]:
diag_mask = np.eye(col_count, dtype=bool)

fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(12, 28))

sns.heatmap(
    class_0_corr,
    mask=diag_mask,
    cmap=sns.diverging_palette(145, 300, s=60, as_cmap=True),
    fmt=".2f",
    cbar_kws={"label": "Correlation Coefficient"},
    ax=axes[0],
)
axes[0].set_title("Class 0 Correlation Matrix", fontsize=20)

sns.heatmap(
    class_1_corr,
    mask=diag_mask,
    cmap=sns.diverging_palette(145, 300, s=60, as_cmap=True),
    fmt=".2f",
    cbar_kws={"label": "Correlation Coefficient"},
    ax=axes[1],
)
axes[1].set_title("Class 1 Correlation Matrix", fontsize=20)

sns.heatmap(
    class_1_corr - class_0_corr,
    mask=diag_mask,
    cmap=sns.diverging_palette(145, 300, s=60, as_cmap=True),
    fmt=".2f",
    cbar_kws={"label": "Correlation Coefficient Difference"},
    ax=axes[2],
)
axes[2].set_title("Diff Correlation Matrix", fontsize=20)

fig.suptitle("Fig 1: Correlation Matrices by Classes", fontsize=20, y=0.92)

### Distributions via Box Plots


In [ ]:
cols = 5
rows = int(np.round(col_count / cols))

fig, axes = plt.subplots(nrows=rows, ncols=cols, figsize=(20, 24))

for i, col in enumerate(features.columns):
    ax = axes[i // cols, i % cols]
    sns.boxplot(
        standardized_features,
        x=col,
        palette="Set2",
        hue=transactions[target_name],
        ax=ax,
    )
    ax.set_title(col)
    ax.set_xlim(-8, 8)

fig.suptitle("Boxplots of Standardized Features by Class", fontsize=20, y=1.02)
fig.tight_layout()

### Clusters


In [ ]:
for group in tuple(itertools.batched(features.columns, 6)):
    p = sns.pairplot(
        sampled_transactions,
        palette="Set1",
        hue=target_name,
        vars=group,
        plot_kws={"alpha": 1, "s": 8},
    )
    p.figure.suptitle(f"Features: {', '.join(group)}", y=1.02)